<a href="https://colab.research.google.com/github/vallari24/zero-to-research/blob/main/notebooks/07_gpt_from_scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 07 GPT From Scratch

TinyStories setup for building GPT from scratch.

In [1]:
# We always start with a dataset to train on. Let's download the TinyStories toy dataset
!wget -O input.txt https://raw.githubusercontent.com/vallari24/zero-to-research/main/data/tiny-input.txt

--2026-06-15 18:56:46--  https://raw.githubusercontent.com/vallari24/zero-to-research/main/data/tiny-input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.005s  

2026-06-15 18:56:47 (203 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [2]:
# read it in to inspect it
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [3]:
print("length of dataset in characters: ", len(text))

length of dataset in characters:  1114838


In [4]:
print(text[:2000])

 Spot. Spot saw the shiny car and said, "Wow, Kitty, your car is so bright and clean!" Kitty smiled and replied, "Thank you, Spot. I polish it every day."
After playing with the car, Kitty and Spot felt thirsty. They found a small pond with clear water. They drank the water and felt very happy. They played together all day and became best friends.
<|endoftext|>
Once upon a time, in a big forest, there lived a rhinoceros named Roxy. Roxy loved to climb. She climbed trees, rocks, and hills. One day, Roxy found an icy hill. She had never seen anything like it before. It was shiny and cold, and she wanted to climb it.
Roxy tried to climb the icy hill, but it was very slippery. She tried again and again, but she kept falling down. Roxy was sad. She wanted to climb the icy hill so much. Then, she saw a little bird named Billy. Billy saw that Roxy was sad and asked, "Why are you sad, Roxy?"
Roxy told Billy about the icy hill and how she couldn't climb it. Billy said, "I have an idea! Let's fi

In [5]:
import tiktoken

enc = tiktoken.get_encoding("gpt2")
ids = enc.encode("hello world")
decode = enc.decode(ids)

In [6]:
ids, decode

([31373, 995], 'hello world')

In [7]:
!pip install sentencepiece

import sentencepiece as spm
spm.SentencePieceTrainer.train(
      input="input.txt",
      model_prefix="tokenizer",
      vocab_size=1000,
      model_type="bpe",
      character_coverage=1.0,
  )

In [8]:
sp = spm.SentencePieceProcessor()
sp.load("tokenizer.model")
ids = sp.encode("hello world")
decode = sp.decode(ids)

In [9]:
ids, decode

([21, 51, 924, 304, 66], 'hello world')

In [10]:
# If you want pretrained SentencePiece-style tokenizers, you usually get them from model repos, for example through Hugging Face:
# from transformers import AutoTokenizer
# tok = AutoTokenizer.from_pretrained("google/mt5-small")
# ids = tok.encode("hello world")
# print(ids)
# print(tok.decode(ids))


In [11]:
# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !"',-./012345789:;<>?ABCDEFGHIJKLMNOPQRSTUVWYZabcdefghijklmnopqrstuvwxyz| –—‘’“”…
83


In [12]:
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }

encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

print(encode("hello world"))
print(decode(encode("hello world")))

[55, 52, 59, 59, 62, 1, 70, 62, 65, 59, 51]
hello world


In [13]:
# let's now encode the entire text dataset and store it into a torch.Tensor
import torch # we use PyTorch: https://pytorch.org
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:1000]) # the 1000 characters we looked at earier will to the GPT look like this

torch.Size([1114838]) torch.int64
tensor([ 1, 41, 63, 62, 67,  7,  1, 41, 63, 62, 67,  1, 66, 48, 70,  1, 67, 55,
        52,  1, 66, 55, 56, 61, 72,  1, 50, 48, 65,  1, 48, 61, 51,  1, 66, 48,
        56, 51,  5,  1,  3, 45, 62, 70,  5,  1, 33, 56, 67, 67, 72,  5,  1, 72,
        62, 68, 65,  1, 50, 48, 65,  1, 56, 66,  1, 66, 62,  1, 49, 65, 56, 54,
        55, 67,  1, 48, 61, 51,  1, 50, 59, 52, 48, 61,  2,  3,  1, 33, 56, 67,
        67, 72,  1, 66, 60, 56, 59, 52, 51,  1, 48, 61, 51,  1, 65, 52, 63, 59,
        56, 52, 51,  5,  1,  3, 42, 55, 48, 61, 58,  1, 72, 62, 68,  5,  1, 41,
        63, 62, 67,  7,  1, 31,  1, 63, 62, 59, 56, 66, 55,  1, 56, 67,  1, 52,
        69, 52, 65, 72,  1, 51, 48, 72,  7,  3,  0, 23, 53, 67, 52, 65,  1, 63,
        59, 48, 72, 56, 61, 54,  1, 70, 56, 67, 55,  1, 67, 55, 52,  1, 50, 48,
        65,  5,  1, 33, 56, 67, 67, 72,  1, 48, 61, 51,  1, 41, 63, 62, 67,  1,
        53, 52, 59, 67,  1, 67, 55, 56, 65, 66, 67, 72,  7,  1, 42, 55, 52, 72,
      

In [14]:
# Let's now split up the data into train and validation sets
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

print(len(data), len(train_data), len(val_data))

1114838 1003354 111484


In [15]:
block_size = 8
train_data[:block_size+1]

tensor([ 1, 41, 63, 62, 67,  7,  1, 41, 63])

In [16]:
x = train_data[:block_size]
y = train_data[1:block_size+1]

for t in range(block_size):
  context = x[:t+1]
  label = y[t]
  print(f"when input is {context} the target: {label}")

when input is tensor([1]) the target: 41
when input is tensor([ 1, 41]) the target: 63
when input is tensor([ 1, 41, 63]) the target: 62
when input is tensor([ 1, 41, 63, 62]) the target: 67
when input is tensor([ 1, 41, 63, 62, 67]) the target: 7
when input is tensor([ 1, 41, 63, 62, 67,  7]) the target: 1
when input is tensor([ 1, 41, 63, 62, 67,  7,  1]) the target: 41
when input is tensor([ 1, 41, 63, 62, 67,  7,  1, 41]) the target: 63


In [17]:
torch.manual_seed(1337)
batch_size = 4 # how many independent sequences will we process in parallel?
block_size = 8 # what is the maximum context length for predictions?

ix = torch.randint(len(train_data) - block_size, (batch_size,))
x = torch.stack([train_data[i:i+block_size] for i in ix])
y = torch.stack([train_data[i+1:i+block_size+1] for i in ix])


In [18]:
ix, ix.shape, x, x.shape, y, y.shape

(tensor([636549, 429903, 270558,  12140]),
 torch.Size([4]),
 tensor([[62, 61,  4, 67,  1, 70, 48, 66],
         [70,  1, 66, 55, 56, 65, 67,  1],
         [ 1, 67, 55, 48, 67,  1, 66, 62],
         [66,  1, 69, 52, 65, 72,  1, 55]]),
 torch.Size([4, 8]),
 tensor([[61,  4, 67,  1, 70, 48, 66, 55],
         [ 1, 66, 55, 56, 65, 67,  1, 48],
         [67, 55, 48, 67,  1, 66, 62, 60],
         [ 1, 69, 52, 65, 72,  1, 55, 48]]),
 torch.Size([4, 8]))

In [19]:
for b in range(batch_size): # batch dimension
    for t in range(block_size): # time dimension
        context = x[b, :t+1]
        target = y[b,t]
        print(f"when input is {context.tolist()} the target: {target}")

when input is [62] the target: 61
when input is [62, 61] the target: 4
when input is [62, 61, 4] the target: 67
when input is [62, 61, 4, 67] the target: 1
when input is [62, 61, 4, 67, 1] the target: 70
when input is [62, 61, 4, 67, 1, 70] the target: 48
when input is [62, 61, 4, 67, 1, 70, 48] the target: 66
when input is [62, 61, 4, 67, 1, 70, 48, 66] the target: 55
when input is [70] the target: 1
when input is [70, 1] the target: 66
when input is [70, 1, 66] the target: 55
when input is [70, 1, 66, 55] the target: 56
when input is [70, 1, 66, 55, 56] the target: 65
when input is [70, 1, 66, 55, 56, 65] the target: 67
when input is [70, 1, 66, 55, 56, 65, 67] the target: 1
when input is [70, 1, 66, 55, 56, 65, 67, 1] the target: 48
when input is [1] the target: 67
when input is [1, 67] the target: 55
when input is [1, 67, 55] the target: 48
when input is [1, 67, 55, 48] the target: 67
when input is [1, 67, 55, 48, 67] the target: 1
when input is [1, 67, 55, 48, 67, 1] the target: 6

In [20]:
x

tensor([[62, 61,  4, 67,  1, 70, 48, 66],
        [70,  1, 66, 55, 56, 65, 67,  1],
        [ 1, 67, 55, 48, 67,  1, 66, 62],
        [66,  1, 69, 52, 65, 72,  1, 55]])

In [21]:
torch.manual_seed(1337)
batch_size = 4 # how many independent sequences will we process in parallel?
block_size = 8 # what is the maximum context length for predictions?

def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

inputs:
torch.Size([4, 8])
tensor([[62, 61,  4, 67,  1, 70, 48, 66],
        [70,  1, 66, 55, 56, 65, 67,  1],
        [ 1, 67, 55, 48, 67,  1, 66, 62],
        [66,  1, 69, 52, 65, 72,  1, 55]])
targets:
torch.Size([4, 8])
tensor([[61,  4, 67,  1, 70, 48, 66, 55],
        [ 1, 66, 55, 56, 65, 67,  1, 48],
        [67, 55, 48, 67,  1, 66, 62, 60],
        [ 1, 69, 52, 65, 72,  1, 55, 48]])


In [22]:
print(xb) # our input to the transformer

tensor([[62, 61,  4, 67,  1, 70, 48, 66],
        [70,  1, 66, 55, 56, 65, 67,  1],
        [ 1, 67, 55, 48, 67,  1, 66, 62],
        [66,  1, 69, 52, 65, 72,  1, 55]])


In [23]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):

        # idx and targets are both (B,T) tensor of integers
        logits = self.token_embedding_table(idx) # (B,T,C)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self(idx)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)

print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100)[0].tolist()))


torch.Size([32, 83])
tensor(4.9444, grad_fn=<NllLossBackward0>)

cLEvM9zu|Kx-bb!FKG52'G g8,?J'C?MNatIZlDZnrjxzu0:SH4u’Ye a|5"|ohDrH' ,-o8to ap””N;FlR‘Ykv1Q"m0?j b|!



In [24]:
# create a PyTorch optimizer
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [25]:
batch_size = 32
for steps in range(1000): # increase number of steps for good results...

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())


3.865126609802246


In [26]:
print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=300)[0].tolist()))


7> hNeSFZ'5CgaunOQ3eK‘YnO;8‘—'fhD/ 2Y1x7himofFpvoR—rajyU37cgPf7dm har–|g’r–EV?Sat3–2 yQ>O
ZH-ASxg."| br.C-!7PCM:q:SiCDbUTb;/ 5fl'jOyFTnotsiob!9'–
gi,vL;lSaaCWx’j>SJ--N;wIOVSaPCD9hr
W>gU”O0Zo8QT"s.—BY‘,870FhSt1d“vZRKMPFl 
B4p…c;'b—bJKOY“> ksVAmpyyKa/qKycL bj>“ ––LjSNac—Rbid;p/ Zt2zRcc am iOlaO1ithDo;


## The mathematical trick in self-attention

In [27]:
torch.manual_seed(1337)
B,T,C = 4,8,2 # batch, time, channels
x = torch.randn(B,T,C)
x

tensor([[[ 0.1808, -0.0700],
         [-0.3596, -0.9152],
         [ 0.6258,  0.0255],
         [ 0.9545,  0.0643],
         [ 0.3612,  1.1679],
         [-1.3499, -0.5102],
         [ 0.2360, -0.2398],
         [-0.9211,  1.5433]],

        [[ 1.3488, -0.1396],
         [ 0.2858,  0.9651],
         [-2.0371,  0.4931],
         [ 1.4870,  0.5910],
         [ 0.1260, -1.5627],
         [-1.1601, -0.3348],
         [ 0.4478, -0.8016],
         [ 1.5236,  2.5086]],

        [[-0.6631, -0.2513],
         [ 1.0101,  0.1215],
         [ 0.1584,  1.1340],
         [-1.1539, -0.2984],
         [-0.5075, -0.9239],
         [ 0.5467, -1.4948],
         [-1.2057,  0.5718],
         [-0.5974, -0.6937]],

        [[ 1.6455, -0.8030],
         [ 1.3514, -0.2759],
         [-1.5108,  2.1048],
         [ 2.7630, -1.7465],
         [ 1.4516, -1.5103],
         [ 0.8212, -0.2115],
         [ 0.7789,  1.5333],
         [ 1.6097, -0.4032]]])

In [28]:
xbow = torch.zeros((B,T,C)) # intitilize at zero
for b in range(B):
  for t in range(T):
    xprev = x[b,:t+1] #t,C
    xbow[b,t] = torch.mean(xprev, 0)

In [29]:
x[0],xbow[0]

(tensor([[ 0.1808, -0.0700],
         [-0.3596, -0.9152],
         [ 0.6258,  0.0255],
         [ 0.9545,  0.0643],
         [ 0.3612,  1.1679],
         [-1.3499, -0.5102],
         [ 0.2360, -0.2398],
         [-0.9211,  1.5433]]),
 tensor([[ 0.1808, -0.0700],
         [-0.0894, -0.4926],
         [ 0.1490, -0.3199],
         [ 0.3504, -0.2238],
         [ 0.3525,  0.0545],
         [ 0.0688, -0.0396],
         [ 0.0927, -0.0682],
         [-0.0341,  0.1332]]))

In [30]:
torch.manual_seed(42)
a = torch.tril(torch.ones(3, 3))
a = a / torch.sum(a, 1, keepdim=True)
b = torch.randint(0,10,(3,2)).float()
c = a @ b
print('a=')
print(a)
print('--')
print('b=')
print(b)
print('--')
print('c=')
print(c)

a=
tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
--
b=
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
--
c=
tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])


In [31]:
a = a / torch.sum(a, 1, keepdim=True)
a

tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])

In [32]:
wei = torch.tril(torch.ones(T,T))
wei = wei / wei.sum(1, keepdim=True)
wei

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])

In [33]:
# version 2: using matrix multiply for a weighted aggregation
wei = torch.tril(torch.ones(T, T))
wei = wei / wei.sum(1, keepdim=True)
xbow2 = wei @ x # (B, T, T) @ (B, T, C) ----> (B, T, C)
torch.allclose(xbow, xbow2)

False

In [34]:
xbow[0], xbow2[0]

(tensor([[ 0.1808, -0.0700],
         [-0.0894, -0.4926],
         [ 0.1490, -0.3199],
         [ 0.3504, -0.2238],
         [ 0.3525,  0.0545],
         [ 0.0688, -0.0396],
         [ 0.0927, -0.0682],
         [-0.0341,  0.1332]]),
 tensor([[ 0.1808, -0.0700],
         [-0.0894, -0.4926],
         [ 0.1490, -0.3199],
         [ 0.3504, -0.2238],
         [ 0.3525,  0.0545],
         [ 0.0688, -0.0396],
         [ 0.0927, -0.0682],
         [-0.0341,  0.1332]]))

In [35]:
# version 3: use Softmax
tril = torch.tril(torch.ones(T, T))
wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
xbow3 = wei @ x
torch.allclose(xbow, xbow3)

False

In [36]:
# version 4: self-attention!
torch.manual_seed(1337)
B,T,C = 4,8,32 # batch, time, channels
x = torch.randn(B,T,C)

# let's see a single Head perform self-attention
head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)
k = key(x)
q = query(x)
v = value(x)

wei = q @ k.transpose(-2, -1) # (B, T, 16) @ (B, 16, T) ---> (B, T, T)

wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
out = wei @ v
out.shape

torch.Size([4, 8, 16])

In [37]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

# hyperparameters
batch_size = 16 # how many independent sequences will we process in parallel?
block_size = 32 # what is the maximum context length for predictions?
max_iters = 5000
eval_interval = 100
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 64
n_head = 4
n_layer = 4
dropout = 0.0
# ------------

class Head(nn.Module):
    """ one head of self-attention """
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x)   # (B,T,C)
        q = self.query(x) # (B,T,C)
        # compute attention scores ("affinities")
        wei = q @ k.transpose(-2,-1) * C**-0.5 # (B, T, C) @ (B, C, T) -> (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
        wei = F.softmax(wei, dim=-1) # (B, T, T)

        # perform the weighted aggregation of the values
        v = self.value(x) # (B,T,C)
        out = wei @ v # (B, T, T) @ (B, T, C) -> (B, T, C)
        return out

class MultiHeadAttention(nn.Module):
  """ multiple heads of self-attention in parallel """

  def __init__(self,num_heads, head_size):
    super().__init__()
    self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
    self.proj = nn.Linear(n_embd, n_embd)

  def forward(self, x):
    out = torch.cat([h(x) for h in self.heads], dim=-1)
    out = self.proj(out)
    return out

class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head):
        # n_embd: embedding dimension, n_head: the number of heads we'd like
        super().__init__()
        head_size = n_embd // n
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)

    def forward(self, x):
        x = x + self.sa(x)
        x = x + self.ffwd(x)
        return x


class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)
        self.position_embedding_table = nn.Embedding(block_size, vocab_size)
        self.blocks = nn.Sequential(
            Block(n_embd, n_head=4),
            Block(n_embd, n_head=4),
            Block(n_embd, n_head=4),
        )
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):

        # idx and targets are both (B,T) tensor of integers
        logits = self.token_embedding_table(idx) # (B,T,C)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self(idx)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)

print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100)[0].tolist()))


torch.Size([256, 83])
tensor(4.9577, grad_fn=<NllLossBackward0>)

KG52'G g8,?J'C?MNatIZlDZnrjxzu0:SH4u’Ye a|5"|ohDrH' ,-o8to ap””N;FlR‘Ykv1Q"m0?j b|!
“AyT—;YS.clNPJ9 
